<a href="https://colab.research.google.com/github/lilianeuwase2/math_forML-Formative3/blob/main/bayesian_probability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

step 1 : set up and chose keywords

In [ ]:
import csv

FILE = "IMDB Dataset.csv"
TARGET = "positive"

POSITIVE_KEYWORDS = ["excellent", "wonderful", "masterpiece", "loved"]
NEGATIVE_KEYWORDS = ["worst", "waste", "awful", "boring"]
ALL_KEYWORDS = POSITIVE_KEYWORDS + NEGATIVE_KEYWORDS

step 2 : cleaning the reviews

In [ ]:
def clean_words(text):
  text = text.lower()
  letters_only = "".join(ch if ch.isalpha () else " "for ch in text)
  words = letters_only.split()
  return  set(words)


step 3: reading the reviews

In [ ]:
def load_reviews(path):
    reviews = []
    with open(path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            words = clean_words(row["review"])
            label = row["sentiment"]
            reviews.append((words, label))
    return reviews

reviews = load_reviews(FILE)
print("Number of reviews:", len(reviews))
print("Label of the first review:", reviews[0][1])
print("Different words in the first review:", len(reviews[0][0]))

Number of reviews: 50000
Label of the first review: positive
Different words in the first review: 189


Step 4 :The Prior P counting the positive reviews

In [ ]:
TOTAL = len(reviews)
N_POSITIVE = 0

for words, label in reviews:
    if label == TARGET:
        N_POSITIVE += 1
PRIOR = N_POSITIVE / TOTAL
print(f"Total reviews:    {TOTAL}")
print(f"Positive reviews: {N_POSITIVE}")
print(f"Prior P(Positive) = {N_POSITIVE}/{TOTAL} = {PRIOR:.4f}")

Total reviews:    50000
Positive reviews: 25000
Prior P(Positive) = 25000/50000 = 0.5000


Step 5: counting the keywords

In [ ]:
n_keyword = {}
n_both = {}
for word in ALL_KEYWORDS:
    n_keyword[word] = 0
    n_both[word] = 0

for words, label in reviews:
    for word in ALL_KEYWORDS:
        if word in words:
            n_keyword[word] += 1
            if label == TARGET:
                n_both[word] += 1

for word in ALL_KEYWORDS:
    print(f"{word:<12} in {n_keyword[word]} reviews, {n_both[word]} of them positive")

excellent    in 3552 reviews, 2868 of them positive
wonderful    in 2780 reviews, 2258 of them positive
masterpiece  in 1207 reviews, 878 of them positive
loved        in 2537 reviews, 1912 of them positive
worst        in 4434 reviews, 411 of them positive
waste        in 2534 reviews, 175 of them positive
awful        in 2883 reviews, 284 of them positive
boring       in 3052 reviews, 592 of them positive


step 6 : Baye's theorem for every keyword

In [ ]:
def bayes(word):
    if n_keyword[word] == 0:
        return None
    likelihood = n_both[word] / N_POSITIVE
    marginal = n_keyword[word] / TOTAL
    posterior = (likelihood * PRIOR) / marginal
    return {"keyword": word, "n_keyword": n_keyword[word], "n_both": n_both[word],
            "prior": PRIOR, "likelihood": likelihood,
            "marginal": marginal, "posterior": posterior}


def show_result(r):
    k = r["keyword"]
    print(f"Keyword: '{k}'  (in {r['n_keyword']} reviews, {r['n_both']} of them positive)")
    print(f"  Prior       P(Positive)       = {N_POSITIVE}/{TOTAL} = {r['prior']:.4f}")
    print(f"  Likelihood  P({k} | Positive) = {r['n_both']}/{N_POSITIVE} = {r['likelihood']:.4f}")
    print(f"  Marginal    P({k})            = {r['n_keyword']}/{TOTAL} = {r['marginal']:.4f}")
    print(f"  Posterior   P(Positive | {k}) = {r['likelihood']:.4f} x {r['prior']:.4f} / {r['marginal']:.4f} = {r['posterior']:.4f}")
    print()


results = []
for word in ALL_KEYWORDS:
    r = bayes(word)
    if r is None:
        print(f"'{word}' never appears in the reviews - choose another word")
    else:
        results.append(r)
        show_result(r)

Keyword: 'excellent'  (in 3552 reviews, 2868 of them positive)
  Prior       P(Positive)       = 25000/50000 = 0.5000
  Likelihood  P(excellent | Positive) = 2868/25000 = 0.1147
  Marginal    P(excellent)            = 3552/50000 = 0.0710
  Posterior   P(Positive | excellent) = 0.1147 x 0.5000 / 0.0710 = 0.8074

Keyword: 'wonderful'  (in 2780 reviews, 2258 of them positive)
  Prior       P(Positive)       = 25000/50000 = 0.5000
  Likelihood  P(wonderful | Positive) = 2258/25000 = 0.0903
  Marginal    P(wonderful)            = 2780/50000 = 0.0556
  Posterior   P(Positive | wonderful) = 0.0903 x 0.5000 / 0.0556 = 0.8122

Keyword: 'masterpiece'  (in 1207 reviews, 878 of them positive)
  Prior       P(Positive)       = 25000/50000 = 0.5000
  Likelihood  P(masterpiece | Positive) = 878/25000 = 0.0351
  Marginal    P(masterpiece)            = 1207/50000 = 0.0241
  Posterior   P(Positive | masterpiece) = 0.0351 x 0.5000 / 0.0241 = 0.7274

Keyword: 'loved'  (in 2537 reviews, 1912 of them positi

step 7: summary table

In [ ]:
print("| Keyword | Type | Prior | Likelihood | Marginal | Posterior |")
print("|---|---|---|---|---|---|")

for r in results:
    if r["keyword"] in POSITIVE_KEYWORDS:
        kind = "positive word"
    else:
        kind = "negative word"
    print(f"| {r['keyword']} | {kind} | {r['prior']:.4f} | {r['likelihood']:.4f} | {r['marginal']:.4f} | {r['posterior']:.4f} |")

| Keyword | Type | Prior | Likelihood | Marginal | Posterior |
|---|---|---|---|---|---|
| excellent | positive word | 0.5000 | 0.1147 | 0.0710 | 0.8074 |
| wonderful | positive word | 0.5000 | 0.0903 | 0.0556 | 0.8122 |
| masterpiece | positive word | 0.5000 | 0.0351 | 0.0241 | 0.7274 |
| loved | positive word | 0.5000 | 0.0765 | 0.0507 | 0.7536 |
| worst | negative word | 0.5000 | 0.0164 | 0.0887 | 0.0927 |
| waste | negative word | 0.5000 | 0.0070 | 0.0507 | 0.0691 |
| awful | negative word | 0.5000 | 0.0114 | 0.0577 | 0.0985 |
| boring | negative word | 0.5000 | 0.0237 | 0.0610 | 0.1940 |


step 8: checking if what we did was correct

In [ ]:
for r in results:
    direct = r["n_both"] / r["n_keyword"]
    print(f"{r['keyword']:<12} Bayes = {r['posterior']:.4f}   counting directly = {direct:.4f}")

excellent    Bayes = 0.8074   counting directly = 0.8074
wonderful    Bayes = 0.8122   counting directly = 0.8122
masterpiece  Bayes = 0.7274   counting directly = 0.7274
loved        Bayes = 0.7536   counting directly = 0.7536
worst        Bayes = 0.0927   counting directly = 0.0927
waste        Bayes = 0.0691   counting directly = 0.0691
awful        Bayes = 0.0985   counting directly = 0.0985
boring       Bayes = 0.1940   counting directly = 0.1940
